In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

### Add mbt-gym to path

In [2]:
import sys
sys.path.append("../")

In [3]:
from mbt_gym.agents.BaselineAgents import CarteaJaimungalMmAgent
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.wrappers import *
from mbt_gym.rewards.RewardFunctions import PnL, CjMmCriterion
from mbt_gym.stochastic_processes.midprice_models import BrownianMotionMidpriceModel
from mbt_gym.stochastic_processes.arrival_models import PoissonArrivalModel
from mbt_gym.stochastic_processes.fill_probability_models import ExponentialFillFunction
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics

### Create market making environment

In [4]:
import sys
sys.path.append("../") # This version of the notebook is in the subfolder "notebooks" of the repo

import gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from copy import deepcopy


from mbt_gym.agents.BaselineAgents import *
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.helpers.plotting import *
from mbt_gym.stochastic_processes.midprice_models import *
from mbt_gym.stochastic_processes.arrival_models import *
from mbt_gym.stochastic_processes.fill_probability_models import *
import torch
#print(torch.cuda.is_available())
#print(torch.cuda.get_device_name())
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics
seed = 42


## Varying fad proportion (paramter q)

### Parameters

In [5]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fill_exponent = 1
fads_proportions = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
baseline_arrival_rate = np.array([[5.0, 5.0]])
phi = 0
psi = 30
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [ ]:
def get_as_env(num_trajectories:int = 1, fads_proportion:float=0.6):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = FadsInformedUniformedTradersArrivalModel(baseline_arrival_rate=baseline_arrival_rate,
                                                step_size=terminal_time/n_steps,
                                                phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma, terminal_time=terminal_time,
                                                num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=fill_exponent, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,                      
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [8]:
results_dict = {}
for fads_proportion in fads_proportions:
    vec_env = get_as_env(num_trajectories=1000, fads_proportion=fads_proportion)

    vec_as = CarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[fads_proportion] = dict(results=results, rewards=total_rewards, obs=observations)

NameError: name 'reward' is not defined

In [ ]:
header = f"{'Fads Prop':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for fads_prop, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{fads_prop:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

 Fads Prop |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       0.0 |      10.39 |       3.64 |          -0.141 | 3.820355873475664
       0.2 |      10.37 |       4.06 |          -0.137 | 3.818144968436898
       0.4 |      10.35 |       4.93 |          -0.146 | 3.8189899188136125
       0.6 |      10.33 |       6.07 |          -0.148 | 3.8215305834181152
       0.8 |      10.31 |       7.32 |          -0.145 | 3.8246535790839937
         1 |      10.30 |       8.58 |           -0.15 | 3.8321664890763816


## Varying eta

### Parameters

In [ ]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fill_exponent = 0
fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
etas = [2.5, 5, 7.5, 10.0, 12.5]
baseline_arrival_rate = np.array([[5.0, 5.0]])
phi = 0
psi = 30
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [ ]:
def get_as_env(num_trajectories:int = 1, eta:float=10):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = FadsInformedUniformedTradersArrivalModel(baseline_arrival_rate=baseline_arrival_rate,
                                                step_size=terminal_time/n_steps,
                                                phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma, terminal_time=terminal_time,
                                                num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=fill_exponent, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [ ]:
results_dict = {}
for eta in etas:
    vec_env = get_as_env(num_trajectories=1000, eta=eta)

    vec_as = CarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[eta] = dict(results=results, rewards=total_rewards, obs=observations)

type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5
DEBUG: self.lambdas = [[5.]
 [5.]
 [5.]
 ...
 [5.]
 [5.]
 [5.]]
DEBUG: type(self.lambdas) = <class 'numpy.ndarray'>
DEBUG: self.lambdas[BID_INDEX] = [5.]
DEBUG: type(self.lambdas[BID_INDEX]) = <class 'numpy.ndarray'>


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:169: RuntimeWarning: divide by zero encountered in log
  return 1 / self.kappa * np.log(omega_function)


KeyboardInterrupt: 

In [ ]:
header = f"{'Eta':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for etas, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{etas:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

       Eta |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       2.5 |      19.88 |      19.32 |          -0.069 | 5.432516820038388
         5 |      20.01 |      12.25 |          -0.047 | 5.428148026721453
       7.5 |      20.01 |       9.29 |          -0.041 | 5.429117699958254
      10.0 |      20.01 |       7.79 |           -0.04 | 5.425532231956604
      12.5 |      20.02 |       6.94 |          -0.032 | 5.424479329852773


## Varying gamma parameter

### Parameters

In [ ]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fill_exponent = 0
fads_proportions = 0.6# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
baseline_arrival_rate = np.array([[5.0, 5.0]])
phi = 0
psi = 30
k = 1
gammas = [0, 1, 2, 3]
alpha=0.001
big_phi=0.1
mu=0

In [ ]:
def get_as_env(num_trajectories:int = 1, gamma:float=1):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = FadsInformedUniformedTradersArrivalModel(baseline_arrival_rate=baseline_arrival_rate,
                                                step_size=terminal_time/n_steps,
                                                phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma, terminal_time=terminal_time,
                                                num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=fill_exponent, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [ ]:
results_dict = {}
for gamma in gammas:
    vec_env = get_as_env(num_trajectories=1000, gamma=gamma)

    vec_as = CarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[gamma] = dict(results=results, rewards=total_rewards, obs=observations)

type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5
type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5
type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5
type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5


In [ ]:
header = f"{'Gamma':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for gamma, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{gamma:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Gamma |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
         0 |      20.05 |       7.80 |          -0.027 | 5.4236768893436125
         1 |      20.01 |       7.79 |           -0.04 | 5.425532231956604
         2 |      19.98 |       7.80 |          -0.042 | 5.424595468788433
         3 |      19.91 |       7.79 |          -0.062 | 5.435821557041768


## Varying Informed trader proportion (psi and phi)

### Parameters

In [ ]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fill_exponent = 0
fads_proportion = 0.6
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
baseline_arrival_rate = np.array([[5.0, 5.0]])
phis = [30, 22.5, 15, 7.5, 0]
psis = [0, 7.5, 15, 22.5, 30]
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [ ]:
def get_as_env(num_trajectories:int = 1, phi:float=15, psi:float=15):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = FadsInformedUniformedTradersArrivalModel(baseline_arrival_rate=baseline_arrival_rate,
                                                step_size=terminal_time/n_steps,
                                                phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma, terminal_time=terminal_time,
                                                num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=fill_exponent, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [ ]:
print(zip(phis, psis))

In [ ]:
results_dict = {}
for phi, psi in zip(phis, psis):
    vec_env = get_as_env(num_trajectories=1000, phi=phi, psi=psi)

    vec_as = CarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[(phi, psi)] = dict(results=results, rewards=total_rewards,  obs=observations)

type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5
type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5
type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5
type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5
type(self.env.model_dynamics.fill_probability_model): <class 'int'> 0
self.kappa from fill_probability_model: 1.5


In [ ]:
header = f"{'Phi':>8} | {'Psi':>8} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for (phi, psi), result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{phi:8} | {psi:8} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Phi |      Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
-------------------------------------------------------------------------------
      30 |        0 |      20.05 |       7.80 |          -0.027 | 5.4236768893436125
    22.5 |      7.5 |      20.03 |       7.80 |          -0.029 | 5.42329779746604
      15 |       15 |      20.04 |       7.81 |          -0.024 | 5.42396755152536
     7.5 |     22.5 |      20.03 |       7.80 |          -0.032 | 5.42669107283619
       0 |       30 |      20.01 |       7.79 |           -0.04 | 5.425532231956604
